# 05 — Cross-Resolution Comparison and Modelling-Resolution Selection

## Purpose

The previous four notebooks examined the SOLETE files separately at:

- 60-minute resolution;
- 5-minute resolution;
- 1-minute resolution; and
- 1-second resolution.

This notebook now compares them directly and answers a practical design
question:

> **Which SOLETE resolution should be used for the main forecasting experiments?**

The decision should not be based simply on choosing the finest available
sampling interval. We compare:

1. temporal completeness and stored row order;
2. availability and behaviour of the two candidate generation targets;
3. numerical consistency between resolutions;
4. evidence about how the published resolution files relate to one another;
5. compatibility with 15-, 30-, and 60-minute forecast horizons; and
6. computational cost.

No source measurement is modified here. Preprocessing decisions are deferred
until the modelling resolution has been selected.


In [1]:
from pathlib import Path
import sys

import h5py
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete

dataset_paths = {
    "60min": PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_60min.h5",
    "5min": PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5",
    "1min": PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_1min.h5",
    "1sec": PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_1sec.h5",
}

missing_files = [str(path) for path in dataset_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing SOLETE files:\n" + "\n".join(missing_files))

# The three smaller files can be handled directly in pandas.
df_60min_raw = load_solete(dataset_paths["60min"])
df_60min = df_60min_raw.sort_index()
df_5min = load_solete(dataset_paths["5min"])
df_1min = load_solete(dataset_paths["1min"])

dataframes = {
    "60min": df_60min,
    "5min": df_5min,
    "1min": df_1min,
}


## Structural comparison

All four files cover the same overall period and contain the same 11 named
variables, but their **stored row order** is not identical.

For time-series analysis we distinguish between:

- **source order** — how rows appear in the HDF5 file;
- **analysis order** — chronological order after sorting by timestamp where
  required.

The 60-minute and 1-second files are not stored globally chronologically,
whereas the 5-minute and 1-minute files are. Earlier EDA established that all
four timestamp grids are nevertheless complete and unique.


In [2]:
expected_intervals = {
    "60min": pd.Timedelta(hours=1),
    "5min": pd.Timedelta(minutes=5),
    "1min": pd.Timedelta(minutes=1),
}

source_dataframes = {
    "60min": df_60min_raw,
    "5min": df_5min,
    "1min": df_1min,
}

summary_rows = []

for resolution in ["60min", "5min", "1min"]:
    source_df = source_dataframes[resolution]
    analysis_df = dataframes[resolution]
    expected_interval = expected_intervals[resolution]

    source_diff = source_df.index.to_series().diff().dropna()
    analysis_diff = analysis_df.index.to_series().diff().dropna()

    summary_rows.append({
        "resolution": resolution,
        "rows": len(source_df),
        "columns": source_df.shape[1],
        "start": analysis_df.index.min(),
        "end": analysis_df.index.max(),
        "missing_values": int(source_df.isna().sum().sum()),
        "duplicate_timestamps": int(source_df.index.duplicated().sum()),
        "source_order_chronological": source_df.index.is_monotonic_increasing,
        "source_order_unexpected_intervals": int(
            (source_diff != expected_interval).sum()
        ),
        "timestamp_grid_complete": bool(
            analysis_df.index.is_unique
            and (analysis_diff == expected_interval).all()
        ),
    })

# Notebook 04 established these complete-file timestamp properties using
# chunked reads of the 1-second HDF5 file.
with h5py.File(dataset_paths["1sec"], "r") as f:
    one_sec_shape = f["DATA"]["block0_values"].shape

summary_rows.append({
    "resolution": "1sec",
    "rows": int(one_sec_shape[0]),
    "columns": int(one_sec_shape[1]),
    "start": pd.Timestamp("2018-06-01 00:00:00", tz="UTC"),
    "end": pd.Timestamp("2019-09-01 00:00:00", tz="UTC"),
    "missing_values": 0,
    "duplicate_timestamps": 0,
    "source_order_chronological": False,
    "source_order_unexpected_intervals": 456,
    "timestamp_grid_complete": True,
})

structural_summary = (
    pd.DataFrame(summary_rows)
    .set_index("resolution")
)

structural_summary


,rows,columns,start,end,missing_values,duplicate_timestamps,source_order_chronological,source_order_unexpected_intervals,timestamp_grid_complete
resolution,,,,,,,,,
60min,10969,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,False,456,True
5min,131617,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,True,0,True
1min,658081,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,True,0,True
1sec,39484801,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,False,456,True


## One-pass 1-second reconstruction and target summary

The largest computational task in this notebook is the 1-second file. We scan
it **once** in chunks.

During the same pass we:

- verify the 1-second measurement matrix has no missing values;
- calculate descriptive statistics for `P_Solar[kW]` and `P_Gaia[kW]`; and
- average all 11 variables into hourly bins.

The hourly reconstruction lets us test whether the provided 60-minute file is
numerically consistent with aggregation of the 1-second representation.

For angular variables, arithmetic averaging is used only as a numerical
reconstruction diagnostic. It is not the preferred physical treatment of
angles for later modelling.


In [3]:
solete_columns = list(df_60min.columns)
target_names = ["P_Solar[kW]", "P_Gaia[kW]"]
target_indices = {name: solete_columns.index(name) for name in target_names}

start_time = df_60min.index.min()
end_time = df_60min.index.max()
hourly_index = pd.date_range(start_time, end_time, freq="60min")
n_hours = len(hourly_index)

hourly_sums = np.zeros((n_hours, len(solete_columns)), dtype=np.float64)
hourly_valid_counts = np.zeros((n_hours, len(solete_columns)), dtype=np.int64)
hourly_timestamp_counts = np.zeros(n_hours, dtype=np.int64)
missing_counts = np.zeros(len(solete_columns), dtype=np.int64)

target_stats = {
    name: {
        "count": 0,
        "sum": 0.0,
        "sum_sq": 0.0,
        "min": np.inf,
        "max": -np.inf,
        "zero_count": 0,
    }
    for name in target_names
}

start_ns = start_time.value
one_hour_ns = 60 * 60 * 1_000_000_000
chunk_size = 1_000_000

with h5py.File(dataset_paths["1sec"], "r") as f:
    data = f["DATA"]
    timestamps_ds = data["axis1"]
    values_ds = data["block0_values"]

    hdf5_columns = [name.decode("utf-8") for name in data["axis0"][:]]
    assert hdf5_columns == solete_columns

    for start in range(0, len(timestamps_ds), chunk_size):
        end = min(start + chunk_size, len(timestamps_ds))

        timestamps = timestamps_ds[start:end]
        values = values_ds[start:end, :]

        hour_bins = ((timestamps - start_ns) // one_hour_ns).astype(np.int64)

        hourly_timestamp_counts += np.bincount(
            hour_bins,
            minlength=n_hours,
        )

        missing_counts += np.isnan(values).sum(axis=0)

        # Accumulate hourly means for all variables.
        for column_index in range(len(solete_columns)):
            x = values[:, column_index]
            valid = ~np.isnan(x)

            hourly_sums[:, column_index] += np.bincount(
                hour_bins[valid],
                weights=x[valid],
                minlength=n_hours,
            )
            hourly_valid_counts[:, column_index] += np.bincount(
                hour_bins[valid],
                minlength=n_hours,
            )

        # Collect the target statistics during the same file scan.
        for target, column_index in target_indices.items():
            x = values[:, column_index]
            x = x[~np.isnan(x)]

            s = target_stats[target]
            s["count"] += len(x)
            s["sum"] += float(x.sum())
            s["sum_sq"] += float(np.square(x).sum())
            s["min"] = min(s["min"], float(x.min()))
            s["max"] = max(s["max"], float(x.max()))
            s["zero_count"] += int((x == 0).sum())

hourly_means = np.divide(
    hourly_sums,
    hourly_valid_counts,
    out=np.full_like(hourly_sums, np.nan),
    where=hourly_valid_counts > 0,
)

df_1sec_to_60min = pd.DataFrame(
    hourly_means,
    index=hourly_index,
    columns=solete_columns,
)
df_1sec_to_60min.index.name = "Timestamp"

one_sec_target_rows = []
for target, s in target_stats.items():
    mean = s["sum"] / s["count"]
    variance = (s["sum_sq"] - s["count"] * mean**2) / (s["count"] - 1)

    one_sec_target_rows.append({
        "resolution": "1sec",
        "target": target,
        "count": s["count"],
        "mean": mean,
        "std": np.sqrt(max(variance, 0.0)),
        "min": s["min"],
        "max": s["max"],
        "zero_count": s["zero_count"],
        "zero_percent": 100 * s["zero_count"] / s["count"],
        "nonzero_count": s["count"] - s["zero_count"],
    })

print("1-second missing values across all variables:")
display(pd.Series(missing_counts, index=solete_columns, name="missing_values"))

print("Hourly timestamp-count distribution:")
display(pd.Series(hourly_timestamp_counts).value_counts().sort_index())


1-second missing values across all variables:


TEMPERATURE[degC]    0
HUMIDITY[%]          0
WIND_SPEED[m1s]      0
WIND_DIR[deg]        0
GHI[kW1m2]           0
POA Irr[kW1m2]       0
P_Gaia[kW]           0
P_Solar[kW]          0
Pressure[mbar]       0
Azimuth[deg]         0
Elevation[deg]       0
Name: missing_values, dtype: int64

Hourly timestamp-count distribution:


1           1
3600    10968
Name: count, dtype: int64

## Target behaviour and numerical reconstruction

Target availability is important because a resolution can be temporally
complete while still representing a particular variable poorly.

We first compare the distributions of solar and wind generation across all
four files. We then perform two reconstruction tests:

- **1-minute → 5-minute** by arithmetic averaging;
- **1-second → 60-minute** by timestamp-aware hourly averaging.

These tests do not prove the exact upstream processing pipeline. They show
which published files are **numerically consistent with simple aggregation**
of another resolution.


In [4]:
# ---------- Target summary across all resolutions ----------
target_rows = []

for resolution, dataframe in dataframes.items():
    for target in target_names:
        x = dataframe[target]
        zero_count = int((x == 0).sum())

        target_rows.append({
            "resolution": resolution,
            "target": target,
            "count": len(x),
            "mean": x.mean(),
            "std": x.std(),
            "min": x.min(),
            "max": x.max(),
            "zero_count": zero_count,
            "zero_percent": 100 * zero_count / len(x),
            "nonzero_count": len(x) - zero_count,
        })

target_rows.extend(one_sec_target_rows)

target_summary = (
    pd.DataFrame(target_rows)
    .set_index(["target", "resolution"])
    .sort_index()
)

display(target_summary)

# ---------- Helper for reconstruction comparisons ----------
def reconstruction_summary(provided, reconstructed):
    comparison = pd.concat(
        [
            provided.rename("provided"),
            reconstructed.rename("reconstructed"),
        ],
        axis=1,
        join="inner",
    ).dropna()

    difference = (comparison["provided"] - comparison["reconstructed"]).abs()

    strict_matches = np.isclose(
        comparison["provided"],
        comparison["reconstructed"],
        rtol=1e-9,
        atol=1e-9,
    )

    return {
        "aligned_rows": len(comparison),
        "mean_abs_difference": difference.mean(),
        "median_abs_difference": difference.median(),
        "max_abs_difference": difference.max(),
        "correlation": comparison.corr().iloc[0, 1],
        "strict_numeric_match_percent": 100 * strict_matches.mean(),
    }

# ---------- 1-minute -> 5-minute ----------
df_1min_to_5min = (
    df_1min[solete_columns]
    .resample("5min")
    .mean()
)

comparison_1min_5min = pd.DataFrame({
    variable: reconstruction_summary(
        df_5min[variable],
        df_1min_to_5min[variable],
    )
    for variable in solete_columns
}).T

print("Provided 5-minute vs 1-minute averaged to 5 minutes")
display(comparison_1min_5min)

# ---------- 1-second -> 60-minute ----------
comparison_1sec_60min = pd.DataFrame({
    variable: reconstruction_summary(
        df_60min[variable],
        df_1sec_to_60min[variable],
    )
    for variable in solete_columns
}).T

print("Provided 60-minute vs 1-second averaged to 60 minutes")
display(comparison_1sec_60min)


count      mean       std  min        max  \
target      resolution                                                 
P_Gaia[kW]  1min          658081  2.691753  3.434584  0.0  23.478949   
            1sec        39484801  0.016261  0.334323  0.0  12.313000   
            5min          131617  2.694643  3.382534  0.0  23.330538   
            60min          10969  0.016259  0.323335  0.0  10.184867   
P_Solar[kW] 1min          658081  0.988283  1.688230  0.0   8.404550   
            1sec        39484801  1.024471  1.738146  0.0   9.831133   
            5min          131617  0.987980  1.666584  0.0   7.715423   
            60min          10969  1.029960  1.657731  0.0   6.975178   

                        zero_count  zero_percent  nonzero_count  
target      resolution                                           
P_Gaia[kW]  1min            225913     34.329057         432168  
            1sec          39349911     99.658375         134890  
            5min             38812     29.488592          92805  
            60min            10921     99.562403             48  
P_Solar[kW] 1min            272444     41.399767         385637  
            1sec          16758568     42.443086       22726233  
            5min             54303     41.258348          77314  
            60min             4204     38.326192           6765

Provided 5-minute vs 1-minute averaged to 5 minutes


,aligned_rows,mean_abs_difference,median_abs_difference,max_abs_difference,correlation,strict_numeric_match_percent
TEMPERATURE[degC],131617.0,9.579197e-03,0.0,1.783978e+01,0.999048,99.653540
HUMIDITY[%],131617.0,8.761809e-04,0.0,8.000000e-01,0.988202,99.836647
WIND_SPEED[m1s],131617.0,3.451761e-03,0.0,9.680528e+00,0.998443,99.576802
WIND_DIR[deg],131617.0,2.135106e-01,0.0,2.849743e+02,0.996198,99.577562
GHI[kW1m2],131617.0,2.959082e-06,0.0,1.303890e-01,0.999997,99.915664
POA Irr[kW1m2],131617.0,2.567012e-06,0.0,1.554173e-01,0.999998,99.886033
P_Gaia[kW],131617.0,2.943376e-03,0.0,9.909903e+00,0.999222,99.880714
P_Solar[kW],131617.0,1.485535e-03,0.0,1.682116e+00,0.999902,79.527721
Pressure[mbar],131617.0,2.771678e+00,0.0,8.000000e+02,0.859642,99.653540
Azimuth[deg],131617.0,2.379751e-15,0.0,2.842171e-14,1.000000,100.000000


Provided 60-minute vs 1-second averaged to 60 minutes


,aligned_rows,mean_abs_difference,median_abs_difference,max_abs_difference,correlation,strict_numeric_match_percent
TEMPERATURE[degC],10969.0,0.143595,0.00000,22.299901,0.982781,97.848482
HUMIDITY[%],10969.0,0.013121,0.00000,1.799750,0.844891,97.857599
WIND_SPEED[m1s],10969.0,0.050934,0.00000,12.099708,0.971931,97.848482
WIND_DIR[deg],10969.0,3.187205,0.00000,405.913272,0.932551,97.857599
GHI[kW1m2],10969.0,0.000026,0.00000,0.130932,0.999979,99.772085
POA Irr[kW1m2],10969.0,0.000024,0.00000,0.068158,0.999990,99.544170
P_Gaia[kW],10969.0,0.000000,0.00000,0.000000,1.000000,100.000000
P_Solar[kW],10969.0,0.005775,0.00007,0.538096,0.999953,31.680190
Pressure[mbar],10969.0,41.651295,0.00000,1999.444444,0.228387,95.842830
Azimuth[deg],10969.0,0.000000,0.00000,0.000000,1.000000,100.000000


## Interpretation of the reconstruction evidence

The results do **not** support treating the four files as one simple chain such
as `1 sec → 1 min → 5 min → 60 min`.

Instead, two strong numerical relationships appear:

- the provided **5-minute** file is very close to arithmetic aggregation of the
  **1-minute** file for most variables;
- several variables in the provided **60-minute** file are extremely close to,
  and in some cases exactly reproduced by, hourly aggregation of the
  **1-second** file.

The wind-power target makes this distinction especially clear. `P_Gaia[kW]`
has similar behaviour at 1-minute and 5-minute resolution, whereas it is
almost entirely zero in both the 1-second and 60-minute files. Hourly averaging
of the 1-second Gaia series reproduces the provided 60-minute Gaia series
exactly.

This is evidence of distinct numerical representation patterns across the
published files. It is safer to describe them as **aggregation-consistency
patterns** rather than claiming an undocumented processing pipeline.

The 1-minute and 5-minute files therefore form the strongest pair of
candidates for multivariable forecasting.


In [5]:
# The planned forecasting task evaluates 15-, 30-, and 60-minute horizons.
forecast_horizons_min = [15, 30, 60]

sampling_seconds = {
    "60min": 60 * 60,
    "5min": 5 * 60,
    "1min": 60,
    "1sec": 1,
}

horizon_rows = []

for resolution, interval_seconds in sampling_seconds.items():
    row = {
        "resolution": resolution,
        "observations": {
            "60min": len(df_60min),
            "5min": len(df_5min),
            "1min": len(df_1min),
            "1sec": int(one_sec_shape[0]),
        }[resolution],
    }

    for horizon_min in forecast_horizons_min:
        horizon_seconds = horizon_min * 60

        if horizon_seconds % interval_seconds == 0:
            row[f"{horizon_min}min_horizon_steps"] = (
                horizon_seconds // interval_seconds
            )
        else:
            row[f"{horizon_min}min_horizon_steps"] = "not on native grid"

    horizon_rows.append(row)

horizon_compatibility = (
    pd.DataFrame(horizon_rows)
    .set_index("resolution")
)

display(horizon_compatibility)

candidate_scorecard = pd.DataFrame(
    {
        "criterion": [
            "Observations",
            "Complete regular grid",
            "Chronological source order",
            "PV target well populated",
            "Wind target well populated",
            "15/30/60 min horizons on native grid",
            "Relative computational burden",
        ],
        "1-minute": [
            f"{len(df_1min):,}",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Higher",
        ],
        "5-minute": [
            f"{len(df_5min):,}",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Lower",
        ],
    }
)

candidate_scorecard


,observations,15min_horizon_steps,30min_horizon_steps,60min_horizon_steps
resolution,,,,
60min,10969,not on native grid,not on native grid,1
5min,131617,3,6,12
1min,658081,15,30,60
1sec,39484801,900,1800,3600


,criterion,1-minute,5-minute
0,Observations,"658,081","131,617"
1,Complete regular grid,Yes,Yes
2,Chronological source order,Yes,Yes
3,PV target well populated,Yes,Yes
4,Wind target well populated,Yes,Yes
5,15/30/60 min horizons on native grid,Yes,Yes
6,Relative computational burden,Higher,Lower


## Resolution selection

The comparison supports **5-minute SOLETE** as the primary modelling
resolution.

### Why not 60 minutes?

The 60-minute file is complete after chronological sorting, but it has only
10,969 observations and cannot represent 15- or 30-minute forecast horizons
directly on its native grid. Its Gaia wind-power representation also follows
the sparse behaviour seen in the 1-second file rather than the much denser
wind-power representation available at 1-minute and 5-minute resolution.

### Why not 1 second?

The 1-second file is temporally complete, but it contains about 39.5 million
observations, is stored in shuffled daily-sized blocks, and represents several
important variables very sparsely. For the planned 15-, 30-, and 60-minute
horizons, those horizons would correspond to 900, 1,800, and 3,600 one-second
steps respectively. The computational cost is therefore difficult to justify
for the main multivariable experiments.

### Why 5 minutes rather than 1 minute?

Both are structurally clean and both represent the solar and wind targets
well. The 5-minute file is also numerically very close to simple 5-minute
aggregation of the 1-minute file for most variables.

The difference is computational efficiency:

- 1-minute: **658,081 observations**
- 5-minute: **131,617 observations**

Thus the 1-minute dataset is about five times larger. For the planned
forecast horizons:

- 1-minute data require **15, 30, and 60 steps**;
- 5-minute data require only **3, 6, and 12 steps**.

The 5-minute resolution therefore preserves useful short-term structure while
making repeated model training, comparison, and validation substantially more
manageable.

This is a **data-design decision**, not evidence that 5-minute data will always
produce lower forecast error than 1-minute data. A later sensitivity analysis
can revisit 1-minute resolution if needed.

## Selected strategy

**Primary modelling resolution**
- `SOLETE_Pombo_5min.h5`

**Secondary resolution for sensitivity analysis**
- `SOLETE_Pombo_1min.h5`

**Cross-resolution / diagnostic roles**
- `SOLETE_Pombo_60min.h5`
- `SOLETE_Pombo_1sec.h5`

The next notebook can now focus on preprocessing the selected 5-minute data:
identifying and justifying the transformations required before feature
engineering and forecasting.
